# Notebook 01 — Data Collection, Pre-processing & Cleaning

**Source:** `martj42/international-football-results` (GitHub) — ~47 000 senior international matches since 1872.

**Steps:**
1. Install extra dependencies
2. Download `results.csv` and `shootouts.csv`
3. Inspect raw data
4. Clean: parse dates, handle nulls, standardise team names
5. Engineer structural columns: outcome, goal stats, tournament weight
6. Save cleaned files to `data/processed/`

## 0. Install extra dependencies

In [1]:
import subprocess

result = subprocess.run(
    ['pip', 'install', 'xgboost', 'seaborn', 'kaggle', '--quiet'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('xgboost, seaborn, kaggle installed successfully.')
else:
    print(result.stderr)

xgboost, seaborn, kaggle installed successfully.


## 1. Imports

In [2]:
import os
import requests
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')

pandas  3.0.3
numpy   2.4.4


## 2. Download Raw Data

**Dataset:** `martj42/international-football-results-from-1872-to-2017` on Kaggle (~47 000 matches).

**One-time Kaggle setup** (free account required):
1. Sign in at [kaggle.com](https://www.kaggle.com) → Profile (top-right) → **Settings** → **API** → **Create New Token**
2. A file `kaggle.json` will download to your machine
3. Move it to `C:\Users\shata\.kaggle\kaggle.json`
4. Run the cell below — it downloads and unzips automatically

> If the files are already in `data/raw/` the cell skips the download.

In [3]:
from pathlib import Path

RAW_DIR  = Path('data/raw')
PROC_DIR = Path('data/processed')
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

NEEDED = ['results.csv', 'shootouts.csv']
already_have = all((RAW_DIR / f).exists() for f in NEEDED)

if already_have:
    print('Files already cached:')
    for f in NEEDED:
        size = (RAW_DIR / f).stat().st_size / 1024
        print(f'  {f:22s}  {size:>7.0f} KB')
else:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()

    print('Downloading via Kaggle API ...')
    api.dataset_download_files(
        'martj42/international-football-results-from-1872-to-2017',
        path=str(RAW_DIR),
        unzip=True,
        quiet=False
    )
    print()
    print('Files in data/raw/:')
    for f in NEEDED:
        p = RAW_DIR / f
        if p.exists():
            print(f'  {f:22s}  {p.stat().st_size / 1024:>7.0f} KB  OK')
        else:
            print(f'  {f:22s}  NOT FOUND — check dataset contents on Kaggle')

Dataset URL: https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017


  0%|          | 0.00/1.21M [00:00<?, ?B/s]

 83%|████████▎ | 1.00M/1.21M [00:01<00:00, 687kB/s]

100%|██████████| 1.21M/1.21M [00:01<00:00, 797kB/s]



Files in data/raw/:
  results.csv                3635 KB  OK
  shootouts.csv                28 KB  OK


## 3. Load & Inspect Raw Data

In [4]:
results   = pd.read_csv(RAW_DIR / 'results.csv')
shootouts = pd.read_csv(RAW_DIR / 'shootouts.csv')

print(f'results   : {results.shape[0]:>7,} rows  x  {results.shape[1]} columns')
print(f'shootouts : {shootouts.shape[0]:>7,} rows  x  {shootouts.shape[1]} columns')
print()
print('results columns  :', list(results.columns))
print('shootouts columns:', list(shootouts.columns))
results.head(4)

results   :  49,445 rows  x  9 columns
shootouts :     678 rows  x  5 columns

results columns  : ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
shootouts columns: ['date', 'home_team', 'away_team', 'winner', 'first_shooter']


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False


In [5]:
print('Date range :', results['date'].min(), '->', results['date'].max())
print()
print('Top 10 tournaments by match count:')
print(results['tournament'].value_counts().head(10).to_string())
print()
print('Unique teams (home side):', results['home_team'].nunique())

Date range : 1872-11-30 -> 2026-06-27

Top 10 tournaments by match count:
tournament
Friendly                                18364
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620

Unique teams (home side): 327


In [6]:
print('Shootouts sample:')
shootouts.head(4)

Shootouts sample:


,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN


## 4. Data Cleaning

### 4.1 Parse dates and fix dtypes

In [7]:
results['date']   = pd.to_datetime(results['date'])
results['year']   = results['date'].dt.year
results['month']  = results['date'].dt.month

# neutral is boolean in CSV -> convert to 0/1
results['neutral'] = results['neutral'].astype(int)

results['home_score'] = results['home_score'].astype('Int64')
results['away_score'] = results['away_score'].astype('Int64')

shootouts['date'] = pd.to_datetime(shootouts['date'])

print('Dtypes:')
print(results.dtypes)
print()
print('Year range:', results['year'].min(), '-', results['year'].max())

Dtypes:
date          datetime64[us]
home_team                str
away_team                str
home_score             Int64
away_score             Int64
tournament               str
city                     str
country                  str
neutral                int64
year                   int32
month                  int32
dtype: object

Year range: 1872 - 2026


### 4.2 Handle missing values

In [8]:
print('Nulls in results:')
print(results.isnull().sum())
print()
print('Nulls in shootouts:')
print(shootouts.isnull().sum())

Nulls in results:
date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0
year           0
month          0
dtype: int64

Nulls in shootouts:
date               0
home_team          0
away_team          0
winner             0
first_shooter    422
dtype: int64


In [9]:
before = len(results)
results = results.dropna(subset=['home_score', 'away_score'])
dropped = before - len(results)

results['home_score'] = results['home_score'].astype(int)
results['away_score'] = results['away_score'].astype(int)

print(f'Dropped {dropped} row(s) with missing scores.')
print(f'Remaining: {len(results):,} matches')

Dropped 72 row(s) with missing scores.
Remaining: 49,373 matches


### 4.3 Team name standardisation

Maps historical / alternate names to a single canonical form so ELO ratings are continuous across renames (e.g. West Germany → Germany).

In [10]:
TEAM_NAME_MAP = {
    # Historical → current (for ELO continuity)
    'West Germany'                   : 'Germany',
    'German DR'                      : 'Germany',      # East Germany
    'Soviet Union'                   : 'Russia',

    # Spelling variants
    'Congo DR'                       : 'DR Congo',
    'Democratic Republic of the Congo': 'DR Congo',
    'Zaire'                          : 'DR Congo',
    "Cote d'Ivoire"                  : 'Ivory Coast',
    'Korea Republic'                 : 'South Korea',
    'Korea DPR'                      : 'North Korea',
    'Republic of Ireland'            : 'Ireland',
    'Cape Verde Islands'             : 'Cape Verde',
    'Sao Tome and Principe'          : 'Sao Tome and Principe',
    'Timor-Leste'                    : 'East Timor',
    'Macedonia'                      : 'North Macedonia',
    'China PR'                       : 'China',
}

def standardise(name: str) -> str:
    return TEAM_NAME_MAP.get(str(name).strip(), str(name).strip())

for col in ['home_team', 'away_team']:
    results[col] = results[col].apply(standardise)

for col in ['home_team', 'away_team', 'winner']:
    if col in shootouts.columns:
        shootouts[col] = shootouts[col].apply(standardise)

print(f'Unique teams after standardisation: {results["home_team"].nunique()}')
print('\nSample team names (every 25th alphabetically):')
names = sorted(results['home_team'].unique())
print(names[::25])

Unique teams after standardisation: 326

Sample team names (every 25th alphabetically):
['Abkhazia', 'Basque Country', 'Catalonia', 'Denmark', 'French Guiana', 'Honduras', 'Kurdistan', 'Martinique', 'North Korea', 'Qatar', 'Scotland', 'Switzerland', 'United States', 'Åland Islands']


### 4.4 Outcome column

`outcome`: **0** = home win, **1** = draw, **2** = away win

In [11]:
results['outcome'] = np.where(
    results['home_score'] > results['away_score'], 0,
    np.where(results['home_score'] < results['away_score'], 2, 1)
)

results['goal_diff']   = results['home_score'] - results['away_score']
results['total_goals'] = results['home_score'] + results['away_score']

counts = results['outcome'].value_counts().rename({0: 'Home Win', 1: 'Draw', 2: 'Away Win'}).sort_index()
total  = len(results)
print('Outcome distribution:')
for label, count in counts.items():
    print(f'  {label:10s}: {count:6,}  ({count/total*100:.1f}%)')

Outcome distribution:
  Away Win  : 13,951  (28.3%)
  Draw      : 11,226  (22.7%)
  Home Win  : 24,196  (49.0%)


### 4.5 Tournament importance weight

| Weight | Category |
|--------|----------|
| 5 | FIFA World Cup (final tournament) |
| 4 | Continental championships (Euros, Copa América, AFCON, Asian Cup, Gold Cup) |
| 3 | World Cup qualifiers / continental qualifiers |
| 2 | Nations League, Olympic Games, Confederations Cup |
| 1 | Friendlies and other |


In [12]:
def tournament_weight(t: str) -> int:
    t = str(t).lower()
    if 'world cup' in t and 'qualif' not in t:
        return 5
    if any(x in t for x in ['uefa european', 'copa america', 'africa cup of nations',
                              'afc asian cup', 'concacaf gold cup']):
        return 4
    if 'qualif' in t or 'qualification' in t:
        return 3
    if any(x in t for x in ['nations league', 'olympic', 'confederations cup',
                              'nations cup', 'concacaf nations']):
        return 2
    return 1  # friendlies / other

results['tournament_weight'] = results['tournament'].apply(tournament_weight)
results['is_competitive']    = (results['tournament_weight'] >= 3).astype(int)

print('Matches per weight tier:')
tw = results.groupby('tournament_weight').agg(
    matches=('outcome', 'count'),
    example_tournament=('tournament', lambda x: x.value_counts().index[0])
).sort_index(ascending=False)
print(tw.to_string())

Matches per weight tier:


                   matches            example_tournament
tournament_weight                                       
5                     1024                FIFA World Cup
4                     1250   AFC Asian Cup qualification
3                    15099  FIFA World Cup qualification
2                     1462           UEFA Nations League
1                    30538                      Friendly


### 4.6 Validate coverage — WC 2026 qualified teams

Check that all 48 teams that qualified for WC 2026 appear in the dataset.

In [13]:
# Official WC 2026 qualified teams (48 teams, draw December 2024)
WC2026_TEAMS = [
    # CONCACAF (6)
    'United States', 'Mexico', 'Canada', 'Panama', 'Honduras', 'Jamaica',
    # CONMEBOL (6)
    'Argentina', 'Brazil', 'Colombia', 'Ecuador', 'Uruguay', 'Venezuela',
    # UEFA (16)
    'Germany', 'France', 'Spain', 'England', 'Portugal', 'Netherlands',
    'Belgium', 'Italy', 'Croatia', 'Austria', 'Switzerland', 'Denmark',
    'Serbia', 'Scotland', 'Slovakia', 'Ukraine',
    # CAF (9)
    'Morocco', 'Senegal', 'Egypt', 'Nigeria', 'South Africa', 'DR Congo',
    'Cameroon', 'Mali', 'Algeria',
    # AFC (8)
    'Japan', 'South Korea', 'Iran', 'Australia', 'Saudi Arabia',
    'Jordan', 'Iraq', 'Uzbekistan',
    # OFC (1)
    'New Zealand',
    # Intercontinental playoff winners (2)
    'Kenya', 'Thailand',   # placeholders — update if needed
]

all_teams_in_data = set(results['home_team'].unique()) | set(results['away_team'].unique())
missing = [t for t in WC2026_TEAMS if t not in all_teams_in_data]

if missing:
    print('Teams NOT found in dataset (may need name-map update):')
    for t in missing:
        print(f'  - {t}')
else:
    print('All WC 2026 teams found in dataset.')

All WC 2026 teams found in dataset.


If any teams are missing above, add them to `TEAM_NAME_MAP` and re-run from cell 4.3.

## 5. Final Clean Snapshot

In [14]:
print('Column list after cleaning:')
print(list(results.columns))
print()
results.tail(4)

Column list after cleaning:
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'year', 'month', 'outcome', 'goal_diff', 'total_goals', 'tournament_weight', 'is_competitive']



,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,month,outcome,goal_diff,total_goals,tournament_weight,is_competitive
49369,2026-06-07,Kosovo,Andorra,3,0,Friendly,Pristina,Kosovo,0,2026,6,0,3,3,1,0
49370,2026-06-07,Liechtenstein,Cyprus,0,2,Friendly,Vaduz,Liechtenstein,0,2026,6,2,-2,2,1,0
49371,2026-06-07,Morocco,Norway,1,1,Friendly,Harrison,United States,1,2026,6,1,0,2,1,0
49372,2026-06-07,Denmark,Ukraine,2,1,Friendly,Odense,Denmark,0,2026,6,0,1,3,1,0


In [15]:
print('Basic statistics on goal columns:')
print(results[['home_score', 'away_score', 'goal_diff', 'total_goals']].describe().round(2).to_string())

Basic statistics on goal columns:
       home_score  away_score  goal_diff  total_goals
count    49373.00    49373.00   49373.00     49373.00
mean         1.76        1.18       0.58         2.94
std          1.77        1.40       2.42         2.10
min          0.00        0.00     -21.00         0.00
25%          1.00        0.00      -1.00         1.00
50%          1.00        1.00       0.00         3.00
75%          2.00        2.00       2.00         4.00
max         31.00       21.00      31.00        31.00


## 6. Save Processed Files

In [16]:
results.to_csv(PROC_DIR / 'results_clean.csv', index=False)
shootouts.to_csv(PROC_DIR / 'shootouts_clean.csv', index=False)

print(f'Saved  results_clean.csv   -> {len(results):,} rows')
print(f'Saved  shootouts_clean.csv -> {len(shootouts):,} rows')

Saved  results_clean.csv   -> 49,373 rows
Saved  shootouts_clean.csv -> 678 rows


## 7. Summary

In [17]:
sep = '=' * 52
print(sep)
print('  DATA COLLECTION & CLEANING  —  SUMMARY')
print(sep)
print(f'  Total matches          : {len(results):>8,}')
print(f'  Date range             : {results["year"].min()} – {results["year"].max()}')
print(f'  Unique teams           : {results["home_team"].nunique():>8,}')
print(f'  Unique tournaments     : {results["tournament"].nunique():>8,}')
print(f'  Competitive matches    : {results["is_competitive"].sum():>8,}   (weight >= 3)')
print(f'  World Cup matches      : {(results["tournament_weight"]==5).sum():>8,}')
print(f'  Penalty shootouts      : {len(shootouts):>8,}')
print(sep)
print('  Files written to data/processed/')
print(sep)

  DATA COLLECTION & CLEANING  —  SUMMARY
  Total matches          :   49,373
  Date range             : 1872 – 2026
  Unique teams           :      326
  Unique tournaments     :      200
  Competitive matches    :   17,373   (weight >= 3)
  World Cup matches      :    1,024
  Penalty shootouts      :      678
  Files written to data/processed/
